In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy.integrate as integrate

In [ ]:
'''reading and cleaning data'''

acc_df = pd.read_csv('data/Accelerometer.csv')
gyro_df = pd.read_csv('data/Gyroscope.csv')
mag_df = pd.read_csv('data/Magnetometer.csv')

t_acc=acc_df['Time (s)'].values
t_gyro=gyro_df['Time (s)'].values
t_mag=mag_df['Time (s)'].values

acc_x=acc_df['X (m/s^2)'].values
acc_y=acc_df['Y (m/s^2)'].values
acc_z=acc_df['Z (m/s^2)'].values

raw_gyro_x=gyro_df['X (rad/s)'].values
raw_gyro_y=gyro_df['Y (rad/s)'].values
raw_gyro_z=gyro_df['Z (rad/s)'].values

raw_mag_x=mag_df['X (uT)'].values
raw_mag_y=mag_df['Y (uT)'].values
raw_mag_z=mag_df['Z (uT)'].values

#interpolate to match acc time (master time)
gyro_x=np.interp(t_acc, t_gyro, raw_gyro_x)
gyro_y=np.interp(t_acc, t_gyro, raw_gyro_y)
gyro_z=np.interp(t_acc, t_gyro, raw_gyro_z)

mag_x=np.interp(t_acc, t_mag, raw_mag_x)
mag_y=np.interp(t_acc, t_mag, raw_mag_y)
mag_z=np.interp(t_acc, t_mag, raw_mag_z)

#assuming first 5 ish seconds were "stationary", calibration

station_mask = t_acc < 5.0
offset_acc_x = np.mean(acc_x[station_mask])
offset_acc_y = np.mean(acc_y[station_mask])
offset_acc_z = np.mean(acc_z[station_mask])-9.81

acc_x_calibrated = acc_x - offset_acc_x
acc_y_calibrated = acc_y - offset_acc_y
acc_z_calibrated = acc_z - offset_acc_z

'''positions determined by data'''

vel_x=integrate.cumtrapz(acc_x_calibrated, t_acc, initial=0)
vel_y=integrate.cumtrapz(acc_y_calibrated, t_acc, initial=0)
vel_z=integrate.cumtrapz(acc_z_calibrated, t_acc, initial=0)

pos_x=integrate.cumtrapz(vel_x, t_acc, initial=0)
pos_y=integrate.cumtrapz(vel_y, t_acc, initial=0)
pos_z=integrate.cumtrapz(vel_z, t_acc, initial=0)

In [ ]:
'''gps data'''
gps_df = pd.read_csv('data/Location.csv')
t_gps=gps_df['Time (s)'].values

lat_0=gps_df['Latitude (°)'].iloc[0]
lon_0=gps_df['Longitude (°)'].iloc[0]
h_0=gps_df['Height (m)'].iloc[0]

raw_gps_x = (gps_df['Longitude (°)'].values-lon_0)*111139*np.cos(np.radians(lat_0))
raw_gps_y = (gps_df['Latitude (°)'].values-lat_0)*110574
raw_gps_z = gps_df['Height (m)'].values-h_0

raw_gps_v= gps_df['Velocity (m/s)'].values

gps_x_align= np.interp(t_acc, t_gps, raw_gps_x)
gps_y_align= np.interp(t_acc, t_gps, raw_gps_y)
gps_z_align= np.interp(t_acc, t_gps, raw_gps_z)
gps_v_align= np.interp(t_acc, t_gps, raw_gps_v)


In [ ]:
'''error calculation'''

imu_pos_err= np.sqrt((pos_x-gps_x_align)**2+(pos_y-gps_y_align)**2+(pos_z-gps_z_align)**2)
imu_vel_mag = np.sqrt(vel_x**2+vel_y**2)
imu_vel_err= np.abs(imu_vel_mag-gps_v_align)

quantum_err = np.zeros_like(t_acc) #theoretical zero err for quatum sensor

In [ ]:
'''visualisation'''

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 10))

ax1.plot(t_acc, imu_pos_err, label='IMU Position Error', color='blue')
ax1.plot(t_acc, quantum_err, label='Quantum Position Error', color='red')
ax1.set_title('Total 3D Position Error: IMU vs Quantum PNT')
ax1.set_ylabel('Position Error (meters)')
ax1.legend()
ax1.grid(True)

ax2.plot(t_acc, imu_vel_err, label='IMU Velocity Error', color='blue')
ax2.plot(t_acc, quantum_err, label='Quantum Velocity Error', color='red')
ax2.set_title('Total 3D Velocity Error: IMU vs Quantum PNT')
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Velocity Error (m/s)')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()